### RAG Application using Typesense

In [1]:
import os
from dotenv import load_dotenv
import typesense

if not load_dotenv():
    load_dotenv("../.env")

typesense_host = os.getenv("TYPESENSE_HOST")
typesense_port = os.getenv("TYPESENSE_PORT", "443")
typesense_protocol = os.getenv("TYPESENSE_PROTOCOL", "https")
typesense_api_key = os.getenv("TYPESENSE_API_KEY")

if not all([typesense_host, typesense_api_key]):
    raise ValueError("Missing TYPESENSE_HOST or TYPESENSE_API_KEY in .env file.")

client = typesense.Client({
    "nodes": [{
        "host": typesense_host,
        "port": typesense_port,
        "protocol": typesense_protocol,
    }],
    "api_key": typesense_api_key,
    "connection_timeout_seconds": 2,
})

client

In [2]:
books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'authors', 'type': 'string[]', 'facet': True},
        {'name': 'publication_year', 'type': 'int32', 'facet': True},
        {'name': 'ratings_count', 'type': 'int32'},
        {'name': 'average_rating', 'type': 'float'}
    ],
    'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

In [4]:
with open('../data/json_files/books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()

client.collections['books'].documents.import_(data)

'{"success":true}\n{"success":true}\n{"success":true}\n{"success":true}\n{"success":true}'

In [5]:
search_parameters = {
    'q': 'cleancode',
    'query_by': 'title,authors',
    'sort_by': 'ratings_count:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 3,
 'hits': [{'document': {'authors': ['Robert C. Martin'],
    'average_rating': 4.39,
    'id': '12',
    'publication_year': 2008,
    'ratings_count': 31200,
    'title': 'Clean Code'},
   'highlight': {'title': {'matched_tokens': ['Clean', 'Code'],
     'snippet': '<mark>Clean</mark> <mark>Code</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Clean', 'Code'],
     'snippet': '<mark>Clean</mark> <mark>Code</mark>'}],
   'text_match': 1157451471449491577,
   'text_match_info': {'best_field_score': '2211897872385',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471449491577',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['Robert C. Martin'],
    'average_rating': 4.39,
    'id': '7',
    'publication_year': 2008,
    'ratings_count': 31200,
    'title': 'Clean Code'},
   'highlight': {'title': {'matched_tokens': ['Clean', 'Code'],
     '

In [7]:
search_parameters = {
    'q': 'cleancode',
    'query_by': 'title',
    'filter_by': 'publication_year:<2009',
    'sort_by': 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 3,
 'hits': [{'document': {'authors': ['Robert C. Martin'],
    'average_rating': 4.39,
    'id': '12',
    'publication_year': 2008,
    'ratings_count': 31200,
    'title': 'Clean Code'},
   'highlight': {'title': {'matched_tokens': ['Clean', 'Code'],
     'snippet': '<mark>Clean</mark> <mark>Code</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Clean', 'Code'],
     'snippet': '<mark>Clean</mark> <mark>Code</mark>'}],
   'text_match': 1157451471449491577,
   'text_match_info': {'best_field_score': '2211897872385',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471449491577',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['Robert C. Martin'],
    'average_rating': 4.39,
    'id': '7',
    'publication_year': 2008,
    'ratings_count': 31200,
    'title': 'Clean Code'},
   'highlight': {'title': {'matched_tokens': ['Clean', 'Code'],
     '

In [8]:
### Langchain + Typesense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\sayon\AppData\Local\Temp\ipykernel_38440\210522712.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found in .env file or environment variables.")

loader = TextLoader("../data/text_files/test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

c:\Users\sayon\Documents\Sayon\My Career\Computer Science\Development\AI and ML\Rag\my-rag-with-typesense\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\sayon\Documents\Sayon\My Career\Computer Science\Development\AI and ML\Rag\my-rag-with-typesense\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sayon\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingfac

In [10]:
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import Typesense

if not load_dotenv():
    load_dotenv("../.env")

typesense_host = os.getenv("TYPESENSE_HOST")
typesense_port = os.getenv("TYPESENSE_PORT", "443")
typesense_protocol = os.getenv("TYPESENSE_PROTOCOL", "https")
typesense_api_key = os.getenv("TYPESENSE_API_KEY")
typesense_collection_name = os.getenv("TYPESENSE_COLLECTION_NAME", "rag-docs")

if not all([typesense_host, typesense_api_key]):
    raise ValueError("Missing TYPESENSE_HOST or TYPESENSE_API_KEY in .env file.")

docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": typesense_host,
        "port": typesense_port,
        "protocol": typesense_protocol,
        "typesense_api_key": typesense_api_key,
        "typesense_collection_name": typesense_collection_name,
    },
)

In [11]:
query = "What is Typesense and how is it used in a RAG application?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Retrieval-Augmented Generation (RAG) combines information retrieval with large language models to ground answers in external data. Instead of relying solely on parameters learned during training, a RAG pipeline retrieves relevant passages from a knowledge base at query time and passes them into the LLM prompt.

Typesense is an open-source search engine that supports typo tolerance, filtering, faceting, and vector search. In a RAG application, Typesense can store document chunks and vector embeddings so the system can retrieve relevant context quickly before generating a response with a model hosted on Groq.

Chunking splits long documents into smaller pieces so each piece fits comfortably within embedding and context limits. Using a chunk size with a small overlapâ€”such as 1000 characters with a 100-character overlapâ€”helps preserve context across chunk boundaries when splitting text with CharacterTextSplitter.


In [12]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001BD3A72A180>, search_kwargs={})

In [13]:
query = "What is Typesense and how is it used in a RAG application?"
retriever.invoke(query)[0]

Document(metadata={'source': '../data/text_files/test.txt'}, page_content='Retrieval-Augmented Generation (RAG) combines information retrieval with large language models to ground answers in external data. Instead of relying solely on parameters learned during training, a RAG pipeline retrieves relevant passages from a knowledge base at query time and passes them into the LLM prompt.\n\nTypesense is an open-source search engine that supports typo tolerance, filtering, faceting, and vector search. In a RAG application, Typesense can store document chunks and vector embeddings so the system can retrieve relevant context quickly before generating a response with a model hosted on Groq.\n\nChunking splits long documents into smaller pieces so each piece fits comfortably within embedding and context limits. Using a chunk size with a small overlapâ€”such as 1000 characters with a 100-character overlapâ€”helps preserve context across chunk boundaries when splitting text with CharacterTextSpli